In [24]:
from pyspark.sql import SparkSession
import datetime
import re
import os
import time
import pyspark
from pyspark.sql.types import StructType, StructField
from pyspark.sql.types import StringType
from pyspark.sql.types import DateType
from pyspark.sql.types import IntegerType
from datetime import date
import findspark
findspark.init()

os.environ['PYSPARK_PYTHON'] = 'python'
os.environ['PYSPARK_DRIVER_PYTHON'] ='jupyter'

In [26]:
spark = (
    SparkSession.builder
    .appName("SCD Type Demo")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")

    # ===== ALL PACKAGES IN ONE PLACE =====
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-token-provider-kafka-0-10_2.12:3.5.6",
            "org.postgresql:postgresql:42.7.7"
        ])
    )

    # ===== EXTENSIONS =====
    # .config(
    #     "spark.sql.extensions",
    #     "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
    #     "org.projectnessie.spark.extensions.NessieSparkSessionExtensions"
    # )


    .getOrCreate()
)

spark.conf.set("spark.sql.legacy.parquet.nanosAsLong", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

print("Spark Session Started")

Spark Session Started


In [27]:


jdbc_url = "jdbc:postgresql://localhost:5432/scd_db"
connection_props = {
    "user": "user",
    "password": "password",
    "driver": "org.postgresql.Driver"
}


In [28]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """
CREATE TABLE IF NOT EXISTS scd_example_one (
    id SERIAL PRIMARY KEY,
    name TEXT NOT NULL,
    dob DATE NOT NULL
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


## SCD Type: 0

#### The Type 0 dimension attributes never change and are assigned to attributes that have durable values or are described as 'Original'. Examples: Date of Birth, Original Credit Score. Type 0 applies to most date dimension attributes [Wiki](https://en.wikipedia.org/wiki/Slowly_changing_dimension)

In [30]:

schema_0 = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dob", DateType(), True),

])

data = [

    (1,'Jordan', date(1978,1,1)),
    (2, 'Joe', date(1979,1,1)),
    (3, 'Mack', date(1979,5,1)),
]

df = spark.createDataFrame(data, schema=schema_0)


df.show()

+---+------+----------+
| id|  name|       dob|
+---+------+----------+
|  1|Jordan|1978-01-01|
|  2|   Joe|1979-01-01|
|  3|  Mack|1979-05-01|
+---+------+----------+



In [31]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_one") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

#### With this nothing is updated or upserted to

## SCD Type: 1

#### This method overwrites old with new data, and therefore does not track historical data.

In [34]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """
CREATE TABLE IF NOT EXISTS scd_example_scd_typ_1 (

    player_id VARCHAR(10) NOT NULL PRIMARY KEY,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


In [44]:
schema_1 = StructType([
    StructField("player_id", StringType(), nullable=False),
    StructField("ab", IntegerType(), nullable=False),
    StructField("h", IntegerType(), nullable=False),
    StructField("bb", IntegerType(), nullable=False),
    StructField("so", IntegerType(), nullable=False),
    StructField("rbi", IntegerType(), nullable=False),





])

data = [('60912', 400, 78, 12, 13, 34 ),
        ('60913', 500, 112, 45, 33, 67 ),
        ('60914', 101, 32, 7, 11, 10 ),
        ('60915', 234, 65, 64, 1, 38 )

]


df = spark.createDataFrame(data, schema=schema_1)



df.show()

+---------+---+---+---+---+---+
|player_id| ab|  h| bb| so|rbi|
+---------+---+---+---+---+---+
|    60912|400| 78| 12| 13| 34|
|    60913|500|112| 45| 33| 67|
|    60914|101| 32|  7| 11| 10|
|    60915|234| 65| 64|  1| 38|
+---------+---+---+---+---+---+



In [38]:
df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [45]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+---+---+---+---+---+
|player_id| ab|  h| bb| so|rbi|
+---------+---+---+---+---+---+
|    60912|400| 78| 12| 13| 34|
|    60913|500|112| 45| 33| 67|
|    60915|234| 65| 64|  1| 38|
|    60914|101| 32|  7| 11| 10|
+---------+---+---+---+---+---+



### In SCD Type I, on new updates to the data, the data is updated and the history isn't preserved

#### Unlike, iceberg or delta tables, for postgres this requires a staging table in postgres

In [46]:
new_data = [('60912', 450, 100, 18, 25, 50 ),
        ('60913', 501, 113, 45, 33, 68 ),

]


df = spark.createDataFrame(new_data, schema=schema_1)



df.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1_staging") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()


In [47]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()


upsert_sql = """
INSERT INTO scd_example_scd_typ_1 (player_id, AB, H, BB, SO, RBI)
SELECT player_id, AB, H, BB, SO, RBI
FROM scd_example_scd_typ_1_staging
ON CONFLICT (player_id)
DO UPDATE SET
  AB  = EXCLUDED.AB,
  H  = EXCLUDED.H,
  BB = EXCLUDED.BB,
  SO = EXCLUDED.SO,
  RBI = EXCLUDED.RBI
;
"""

stmt.executeUpdate(upsert_sql)
stmt.close()
conn.close()


In [48]:
df = (
    spark.read
  .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "scd_example_scd_typ_1") \
    .option("user", connection_props["user"]) \
    .option("password", connection_props["password"]) \
    .option("driver", "org.postgresql.Driver") \
    .load()
)

df.show(10)


+---------+---+---+---+---+---+
|player_id| ab|  h| bb| so|rbi|
+---------+---+---+---+---+---+
|    60915|234| 65| 64|  1| 38|
|    60914|101| 32|  7| 11| 10|
|    60912|450|100| 18| 25| 50|
|    60913|501|113| 45| 33| 68|
+---------+---+---+---+---+---+



### From here you can see that the values are able to be updated but the historical past values are completely lost

## SCD Type: 2

#### This method tracks historical data by creating multiple records for a given natural key in the dimensional tables with separate surrogate keys and/or different version numbers. Unlimited history is preserved for each insert.

In [51]:
sc = spark.sparkContext
jvm = sc._jvm


DriverManager = jvm.java.sql.DriverManager
conn = DriverManager.getConnection(jdbc_url, connection_props["user"], connection_props["password"])
stmt = conn.createStatement()

create_sql = """
CREATE TABLE IF NOT EXISTS scd_example_scd_typ_2 (

    player_id VARCHAR(10) NOT NULL PRIMARY KEY,
    AB int NOT NULL,
    H INT NOT NULL,
    BB INT NOT NULL,
    SO INT NOT NULL,
    RBI INT NOT NULL,
    ACTIVE BOOLEAN NOT NULL,
    DATE_REC_CREATED DATE NOT NULL DEFAULT NOW()
);
"""

stmt.executeUpdate(create_sql)
stmt.close()
conn.close()


In [25]:
spark.stop()